# SurgeFlow notebook quickstart

The member beta supports research in `us`, `cn`, `jp`, and `hk`. Research data only, not investment advice.

Set `SURGEFLOW_API_KEY` in the environment, or enter your own key at the hidden prompt. The client sends it only as `Authorization: Bearer`.

The catalogue supplies `base_url`. Set `SURGEFLOW_BASE_URL` to override it. Never place a key in a URL.

The free plan permits 2,000 requests per day and 180 per minute for each key. The monitor defaults to 1,500 per day.

Each panel refreshes on a schedule; each panel shows its source date. Free access grants no redistribution rights. **Do not republish the data you fetch.**

Set `SURGEFLOW_FIXTURES=1` for synthetic offline responses. Fixture mode uses the placeholder key and makes no network request.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_panel, fixture_mode, get_api_key, make_client, nested
client, catalogue = make_client()
api_key = get_api_key()
print('Catalogue status:', catalogue.get('status', 'not supplied'))
print('Base URL:', client.base_url)
print('Fixture mode:', fixture_mode())

In [ ]:
me_columns = {
    'plan': 'The API plan attached to your key.',
    'used_7d': 'Requests recorded during the rolling seven-day usage window.',
    'used_30d': 'Requests recorded during the rolling 30-day usage window.',
    'last_used_at': 'The latest usage time in the batched usage ledger.',
}
def me_rows(payload):
    return [{'plan': payload.get('plan'), **{name: nested(payload, 'usage', name) for name in ('used_7d', 'used_30d', 'last_used_at')}}]
me_frame = fetch_panel(
    'Your plan and usage',
    lambda: client.member_me(api_key),
    me_rows,
    me_columns,
    source_date=lambda p: nested(p, 'usage', 'last_used_at'),
    status=lambda p: 'ok' if p.get('ok') else 'error',
    message=lambda p: nested(p, 'usage', 'note'),
)